In [ ]:
!pip install spikingjelly tonic brian2

# Train LIF

In [ ]:
from spikingjelly.datasets import n_mnist
n_mnist.NMNIST.resource_url_md5()

In [ ]:
from tonic.datasets import nmnist

train_ds = nmnist.NMNIST(save_to='data', train=True)
test_ds  = nmnist.NMNIST(save_to='data', train=False)

  0%|          | 0/1011893601 [00:00<?, ?it/s]

Extracting data/NMNIST/train.zip to data/NMNIST


In [ ]:
import os

old_name = "/content/data/NMNIST/test.zip"
new_name = "/content/data/NMNIST//download/Test.zip"
old_name2 = "/content/data/NMNIST/train.zip"
new_name2 = "/content/data/NMNIST/download/Train.zip"

try:
    os.mkdir("/content/data/NMNIST/download")
    os.rename(old_name, new_name)
    os.rename(old_name2, new_name2)
    print(f"File '{old_name}' successfully renamed to '{new_name}'.")
except FileNotFoundError:
    print(f"Error: File '{old_name}' not found.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from spikingjelly.datasets import n_mnist
from spikingjelly.activation_based import layer, neuron, encoding, functional, surrogate

batch_size = 64

# 1. Load N‑MNIST frames with 64 bins (no need for explicit scaling)
train_ds = n_mnist.NMNIST(root='data/NMNIST', train=True,
                          data_type='frame', frames_number=64, split_by='number')
test_ds  = n_mnist.NMNIST(root='data/NMNIST', train=False,
                          data_type='frame', frames_number=64, split_by='number')
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

Streaming output truncated to the last 5000 lines.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/03032.npz] saved.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/02484.npz] saved.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/01294.npz] saved.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/02211.npz] saved.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/04413.npz] saved.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/01579.npz] saved.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/03604.npz] saved.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/03392.npz] saved.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/00440.npz] saved.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/05845.npz] saved.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/01537.npz] saved.
Frames [data/NMNIST/frames_number_64_split_by_number/test/6/00861.npz] saved.
Frames [data/

In [ ]:
lr = 1e-3
epochs = 5
T = 50
tau = 2.0

In [ ]:
class SNNLIF(nn.Module):
    def __init__(self, tau):
        super().__init__()
        self.layer = nn.Sequential(
            layer.Flatten(),
            layer.Linear(64, 10, bias=False),
            neuron.LIFNode(tau=tau, surrogate_function=surrogate.ATan())
        )

    def forward(self, x):
        return self.layer(x)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = SNNLIF(tau).to(device)

optimizer = torch.optim.Adam(net.parameters(), lr=lr)

def run_epoch(loader, train=True):
    total_loss = total_acc = total = 0
    mode = net.train if train else net.eval
    mode()

    for frames, labels in loader:
        frames, labels = frames.to(device), labels.to(device)
        label_onehot = F.one_hot(labels, 10).float()

        B, T, C, H, W = frames.shape
        frames = frames.view(B * T, C, H, W)
        frames = F.adaptive_avg_pool2d(frames, (8, 4))
        frames = frames.view(B, T, -1)

        optimizer.zero_grad()

        out_spikes = []
        for t in range(T):
            out = net(frames[:, t])
            out_spikes.append(out)

        out_spikes = torch.stack(out_spikes, dim=0)
        out_fr = out_spikes.mean(0)

        loss = F.mse_loss(out_fr, label_onehot)
        if train:
            loss.backward()
            optimizer.step()

        functional.reset_net(net)
        preds = out_fr.argmax(dim=1)
        total_acc += (preds == labels).sum().item()
        total_loss += loss.item() * labels.size(0)
        total += labels.size(0)

    return total_loss / total, total_acc / total

# 5. Training loop
for epoch in range(1, epochs+1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    test_loss, test_acc   = run_epoch(test_loader, train=False)
    print(f"Epoch {epoch:2d} | train: {train_loss:.4f}, {train_acc:.4f} | "
          f"test: {test_loss:.4f}, {test_acc:.4f}")


Epoch  1 | train: 0.0968, 0.2246 | test: 0.0885, 0.4580
Epoch  2 | train: 0.0834, 0.5196 | test: 0.0773, 0.5951
Epoch  3 | train: 0.0752, 0.6082 | test: 0.0713, 0.6511
Epoch  4 | train: 0.0701, 0.6536 | test: 0.0672, 0.6856
Epoch  5 | train: 0.0664, 0.6803 | test: 0.0640, 0.7032


In [ ]:
import numpy as np

net.eval()
weights = net.cpu().layer[1].weight.detach().numpy()
np.savetxt("weights.txt", weights)
read_weights = np.loadtxt("weights.txt")

# HH from LIF

In [ ]:
from brian2 import *
import numpy as np
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

fc1_w = np.loadtxt("weights.txt")
# fc1_b = cpu_model.layer[1].bias.detach().numpy()

@check_units(v=volt, result=Hz)
def alpha_m(v):
    dv = (25*mV - v)
    return clip(dv/(exp(dv/(10*mV)) - 1), -1e6, 1e6) * (0.1/mV) / ms

C_new = 1*uF/cm**2
gNa = 120*msiemens/cm**2
gK = 36*msiemens/cm**2
gL = 0.3*msiemens/cm**2
ENa = 120*mV
EK = -12*mV
EL = -10.5*mV

hh_eqs = '''
dv/dt = (I_inj - INa - IK - IL) / C_new : volt
INa = gNa * m**3 * h * (v - ENa) : amp/meter**2
IK = gK * n**4 * (v - EK) : amp/meter**2
IL = gL * (v - EL) : amp/meter**2

dm/dt = alpham*(1-m) - betam*m : 1
dn/dt = alphan*(1-n) - betan*n : 1
dh/dt = alphah*(1-h) - betah*h : 1

alpham = 0.1/mV*(25*mV - v)/(exp((25*mV - v)/(10*mV)) - 1)/ms : Hz (constant over dt)
betam = 4*exp(-v/(18*mV))/ms : Hz (constant over dt)
alphah = 0.07*exp(-v/(20*mV))/ms : Hz (constant over dt)
betah = 1/(exp((30*mV - v)/(10*mV)) + 1)/ms : Hz (constant over dt)
alphan = 0.01/mV*(10*mV - v)/(exp((10*mV - v)/(10*mV)) - 1)/ms : Hz (constant over dt)
betan = 0.125*exp(-v/(80*mV))/ms : Hz (constant over dt)
I_inj : amp/meter**2
'''


device.reinit()
device.activate()
defaultclock.dt = 0.1*ms

In [ ]:
T=50
def run(index):
    frames, true_label = test_ds[index]

    if not isinstance(frames, torch.Tensor):
        frames = torch.tensor(frames)

    T, C, H, W = frames.shape
    frames = frames.view(T, C, H, W)
    frames = F.adaptive_avg_pool2d(frames, (8, 4))
    frames = frames.view(T, -1)  # shape: [T, 64]

    runtime_per_frame = 100*ms
    run_time = T * runtime_per_frame
    frames = frames.detach().cpu().numpy()
    # Normalize all 64×64 values
    normalized = frames / (np.max(frames) + 1e-9)
    rate_array = (normalized * 1) * Hz  # shape: [64, 64]

    # Use TimedArray to vary input rates over time
    input_rates = TimedArray(rate_array, dt=runtime_per_frame)
    # print(input_rates)
    # for element in rate_array:
    #     print(element)
    input_layer = PoissonGroup(64, rates='input_rates(t, i)')


    # Convert frames to firing rates [0, 200] Hz
    # normalized = frames / (frames.max() + 1e-9)
    # rates = (normalized * 200) * Hz

    # Poisson input layer (64 neurons)
    # input_layer = PoissonGroup(64, rates=rates)

    # Hidden layer: 10 HH neurons
    hidden1 = NeuronGroup(
        10, hh_eqs,
        threshold='v > -55*mV',
        reset='v = -65*mV',
        refractory=3*ms,
        method='exponential_euler'
    )
    hidden1.v = -65*mV
    hidden1.m = 0.05
    hidden1.h = 0.6
    hidden1.n = 0.32
    hidden1.I_inj =0*amp/cm**2

    # Synapses with scaled weights
    scale = (90000000000000) * uA / cm**2
    syn_input_hidden1 = Synapses(input_layer, hidden1, model='w : amp/meter**2', on_pre='I_inj_post += w')
    syn_input_hidden1.connect(True)
    syn_input_hidden1.w = fc1_w.T.flatten() * scale

    # Monitors
    mon_output = SpikeMonitor(hidden1)

    # Create and run network
    net = Network(collect())
    net.run(run_time)

    spike_counts = np.array([mon_output.count[i] for i in range(10)])
    predicted = np.argmax(spike_counts)
    print(spike_counts)
    return predicted, true_label

correct = 0
samples = 1
np.seterr(divide='ignore', invalid='ignore', over='ignore')
import random

for i in range(samples):
    pred, label = run(random.randint(0, len(test_ds)-1))
    print(f"Sample {i}: True = {label}, Pred = {pred}")
    if pred == label:
        correct += 1

print(f"Accuracy: {correct/samples:.2%}")

[ 118  370 2134  198 2134  370 2134  198  370  481]
Sample 0: True = 4, Pred = 2
Accuracy: 0.00%


In [ ]:
print(np.max(np.abs(fc1_w)), np.min(np.abs(fc1_w)))

4.224967002868652 0.006091245915740728


In [ ]:
correct = 0
samples = 10
np.seterr(divide='ignore', invalid='ignore', over='ignore')
import random

for i in range(samples):
    pred, label = run(random.randint(0, len(test_ds)-1))
    print(f"Sample {i}: True = {label}, Pred = {pred}")
    if pred == label:
        correct += 1

print(f"Accuracy: {correct/samples:.2%}")

[2134  162  187  162 2134  162  162  162  162  162]
Sample 0: True = 9, Pred = 0
[  69  296  296 2134 2134 2134 2134 1937  882  672]
Sample 1: True = 0, Pred = 3
[  18  201 2134 2134  260 2134  702  201  201  201]
Sample 2: True = 3, Pred = 2
[2134   78   11 2134   11 2134   11 2134 2134   11]
Sample 3: True = 3, Pred = 0
[2134   70   70   70 2134 2134 2134 2134 2134 2134]
Sample 4: True = 7, Pred = 0
[ 198    1    1    1  198 2134   52 2134 2134 2134]
Sample 5: True = 7, Pred = 5
[  53   53  189 2134   53   53  911   53 2134   53]
Sample 6: True = 1, Pred = 3
[2134 2134 2134 2134 1008 2134 2134  423 2134  423]
Sample 7: True = 2, Pred = 0
[  53  394  232 2134  232 2134   53 1371 2134 1760]
Sample 8: True = 8, Pred = 3
[2134 2134 2134  111 1279 2134   15  111 2134 1279]
Sample 9: True = 1, Pred = 0
Accuracy: 0.00%
